In [0]:
from pyspark.sql import functions as F

In [0]:
base = "/Volumes/fintech_fraud_risk/bronze/vol_bronze"

def autoload_cdc(subfolder_glob, schema_hint_name, table_name):
    (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", True)
        .option("cloudFiles.schemaLocation", f"{base}/_schema/{schema_hint_name}")
        .load(f"{base}/incremental_data/{subfolder_glob}")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .writeStream.format("delta")
        .option("checkpointLocation", f"{base}/_checkpoints/{schema_hint_name}")
        .option("mergeSchema", True)
        .trigger(availableNow=True)
        .toTable(f"fintech_fraud_risk.bronze.{table_name}")
        .awaitTermination()
    )

autoload_cdc("bronze_transactions_*.csv", "bronze_transactions_cdc", "bronze_transaction_cdc")

In [0]:
import os
import shutil
from glob import glob

processed_files = []

# List processed files from the Delta tables
for table in ["bronze_transaction_cdc"]:
    df = spark.read.table(f"fintech_fraud_risk.bronze.{table}")
    files = [row[0] for row in df.select("_source_file").distinct().collect()]
    processed_files.extend(files)

# Move processed files to the new directory (copy then delete from incremental_data)
target_dir = "/Volumes/fintech_fraud_risk/bronze/vol_bronze/Incremental_Processed/"
os.makedirs(target_dir, exist_ok=True)

for file_path in processed_files:
    if os.path.exists(file_path):
        dest = os.path.join(target_dir, os.path.basename(file_path))
        shutil.copy(file_path, dest)
        os.remove(file_path)